# Create Virtual Icechunk Store from Copernicus Marine Black Sea BGC NetCDF files

Uses VirtualiZarr to create an Icechunk store pointing at the first 5 files of
`cmems_mod_blk_bgc-nut_anfc_2.5km_P1D-m` (Nitrate, Phosphate, ODU — daily mean, 3D).

Source files are on Copernicus Marine S3 (`s3.waw3-1.cloudferro.com`), accessed anonymously.
The Icechunk repo is stored on the protocoast S3.

In [ ]:
import warnings
import os
import pandas as pd
import xarray as xr
from dotenv import load_dotenv

import icechunk
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# --- Icechunk store (protocoast S3) ---
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)
storage_endpoint = os.environ['ENDPOINT_URL']
storage_bucket   = 'protocoast-data'
storage_name     = 'blksea-bgc-nut-icechunk-v1'

# --- Source files (Copernicus Marine S3, anonymous) ---
CMEMS_ENDPOINT = 'https://s3.waw3-1.cloudferro.com'
CMEMS_BUCKET   = 'mdl-native-11'
CMEMS_PREFIX   = 'native/BLKSEA_ANALYSISFORECAST_BGC_007_010/cmems_mod_blk_bgc-nut_anfc_2.5km_P1D-m_202511'


In [ ]:
import fsspec
fs = fsspec.filesystem('s3', anon=True, endpoint_url=CMEMS_ENDPOINT)

In [ ]:
listed_files = fs.ls('s3://mdl-native-11/native/BLKSEA_ANALYSISFORECAST_BGC_007_010/cmems_mod_blk_bgc-nut_anfc_2.5km_P1D-m_202511/2024/02/')
listed_files

In [ ]:
listed_files = [f's3://{f}' for f in listed_files]

In [ ]:
listed_files[0]

In [ ]:
xr.open_dataset(fs.open(listed_files[0]))

### Step 1: Set up Icechunk repo with virtual chunk container pointing at Copernicus S3

In [ ]:
# Icechunk store on protocoast
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f'icechunk/{storage_name}',
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()

# NetCDF file chunks live on Copernicus S3 (anonymous)
cmems_s3_prefix = f's3://{CMEMS_BUCKET}/'
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=cmems_s3_prefix,
        store=icechunk.s3_store(
            region='not-used',
            anonymous=True,
            s3_compatible=True,
            force_path_style=True,
            endpoint_url=CMEMS_ENDPOINT,
        ),
    )
)

credentials = icechunk.containers_credentials(
    {cmems_s3_prefix: icechunk.s3_credentials(anonymous=True)}
)

# obstore registry for VirtualiZarr — keyed with s3:// prefix to match listed_files
cmems_store = S3Store(
    bucket=CMEMS_BUCKET,
    endpoint_url=CMEMS_ENDPOINT,
    region='not-used',
    skip_signature=True
)
registry = ObjectStoreRegistry({cmems_s3_prefix: cmems_store})
parser = HDFParser()

### Step 2: Virtualize files and concatenate along time

In [ ]:
print('Virtualizing files...')
vds_list = [
    open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=['time'])
    for url in listed_files
]

ds_virtual = xr.concat(
    vds_list, dim='time', coords='minimal', compat='override', combine_attrs='override'
)
print(ds_virtual)

### Step 3: Write to Icechunk

In [ ]:
import s3fs

icechunk_prefix = f'icechunk/{storage_name}'
fs_proto = s3fs.S3FileSystem(endpoint_url=storage_endpoint)
if fs_proto.exists(f'{storage_bucket}/{icechunk_prefix}'):
    fs_proto.rm(f'{storage_bucket}/{icechunk_prefix}', recursive=True)
    print(f'Deleted existing repo at s3://{storage_bucket}/{icechunk_prefix}')

repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
session = repo.writable_session('main')

print(f'Writing {len(ds_virtual.time)} time steps to Icechunk...')
ds_virtual.virtualize.to_icechunk(session.store)

msg = f'Initialized with {len(listed_files)} BGC-nut files'
session.commit(msg)
print(f'Commit: {msg}')

### Step 4: Verify — open from Icechunk and read actual data

In [ ]:
history = repo.ancestry(branch='main')
latest = next(history)
print(f'Latest commit [{latest.written_at}]: {latest.message}')

session_ro = repo.readonly_session('main')
ds_check = xr.open_zarr(session_ro.store, consolidated=False, chunks={})
print(ds_check)
ds_check

In [ ]:
# Read a surface slice of nitrate to confirm virtual chunk access works
no3_surface = ds_check['no3'].isel(depth=0, time=0).load()
print(no3_surface)
no3_surface.plot()